# ML 05 — Ridge regression baseline

This notebook trains Ridge regression directly from the processed dataset. Preprocessing is fitted inside each training fold using a scikit-learn pipeline, so medians, scaling and one-hot encoding are not learned from held-out rows. The metrics and out-of-fold predictions below are calculated in this run.

In [ ]:
from pathlib import Path
import os

# Works whether Jupyter starts in the repository root or this notebook's folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "data" / "raw").exists() and (p / "Machine Learning Pipeline").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root. Open this notebook from inside the T006 project folder.")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

processed_path = DATA_PROCESSED / "layout_level_dataset.csv"
if not processed_path.exists():
    raise FileNotFoundError("Run ML_01_dataset_understanding_data_dictionary.ipynb first; it builds this processed dataset from raw data.")
df = pd.read_csv(processed_path)
TARGET = "target_U_canopy_mean_m_s"
NUMERIC_FEATURES = ["plan_area_density", "plan_area_density_sq", "height_std_m", "height_max_m", "height_min_m", "height_range_m", "height_skew", "frac_taller_than_mean", "wind_direction_deg"]
CATEGORICAL_FEATURES = ["configuration", "horizontal_configuration", "density_class"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
BASELINE_NUMERIC = ["plan_area_density", "height_mean_m", "height_std_m", "height_max_m", "wind_direction_deg", "building_count"]
BASELINE_FEATURES = BASELINE_NUMERIC + CATEGORICAL_FEATURES
print("Dataset shape:", df.shape)
print("Target:", TARGET)

## 1. Define preprocessing and model

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
])
model = Pipeline([("preprocess", preprocessor), ("ridge", Ridge(alpha=1.0))])
X = df[FEATURES]
y = df[TARGET]
groups = df["family_id"]
cv = GroupKFold(n_splits=5)

## 2. Train and predict each held-out fold

In [ ]:
prediction_rows = []
fold_rows = []
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
    assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
    fold_model = Pipeline([("preprocess", preprocessor), ("ridge", Ridge(alpha=1.0))])
    fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])
    predictions = fold_model.predict(X.iloc[test_idx])
    observed = y.iloc[test_idx].to_numpy()
    fold_rows.append({
        "fold": fold, "n_test_cases": len(test_idx),
        "mae_m_s": mean_absolute_error(observed, predictions),
        "rmse_m_s": np.sqrt(mean_squared_error(observed, predictions)),
        "r2": r2_score(observed, predictions),
    })
    for idx, actual, pred in zip(test_idx, observed, predictions):
        prediction_rows.append({
            "fold": fold, "simulation_id": df.iloc[idx]["simulation_id"],
            "layout_id": df.iloc[idx]["layout_id"], "family_id": df.iloc[idx]["family_id"],
            "observed": float(actual), "predicted": float(pred), "model": "Ridge Regression"
        })

fold_metrics = pd.DataFrame(fold_rows)
predictions_df = pd.DataFrame(prediction_rows)
display(fold_metrics.round(4))
assert predictions_df["simulation_id"].nunique() == len(df), "Every case must be predicted exactly once."

## 3. Calculate pooled out-of-fold metrics

In [ ]:
mae = mean_absolute_error(predictions_df.observed, predictions_df.predicted)
rmse = np.sqrt(mean_squared_error(predictions_df.observed, predictions_df.predicted))
r2 = r2_score(predictions_df.observed, predictions_df.predicted)
summary = pd.DataFrame([{"model": "Ridge Regression", "n_cases": len(predictions_df), "mae_m_s": mae, "rmse_m_s": rmse, "r2": r2}])
display(summary.round(4))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(predictions_df.observed, predictions_df.predicted, alpha=0.8)
low = min(predictions_df.observed.min(), predictions_df.predicted.min())
high = max(predictions_df.observed.max(), predictions_df.predicted.max())
ax.plot([low, high], [low, high], linestyle="--")
ax.set(xlabel="Observed wind speed (m/s)", ylabel="Predicted wind speed (m/s)", title="Ridge: observed vs out-of-fold prediction")
plt.tight_layout()
plt.show()

## 4. Save results generated in this run

In [ ]:
fold_metrics.to_csv(RESULTS / "ridge_fold_metrics.csv", index=False)
predictions_df.to_csv(RESULTS / "ridge_oof_predictions.csv", index=False)
summary.to_csv(RESULTS / "ridge_model_metrics.csv", index=False)
print("Created fresh outputs:")
for path in [RESULTS / "ridge_fold_metrics.csv", RESULTS / "ridge_oof_predictions.csv", RESULTS / "ridge_model_metrics.csv"]:
    print(" -", path)